In [ ]:
!pip install "pillow>=10.0,<12.0" rembg onnxruntime ultralytics roboflow grad-cam gradio 2>&1 | grep -E "(conflict|incompatible|Attempting uninstall)"

  Attempting uninstall: opencv-python-headless
  Attempting uninstall: typer


In [ ]:
import torch
import os
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import cv2
import yaml
import glob
import numpy as np
import matplotlib.pyplot as plt
import random
import torch.nn as nn

from ultralytics import YOLO
from PIL import Image
from roboflow import Roboflow
from torchvision import transforms
from skimage.transform import resize
from tqdm import tqdm

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
SEED = 42

In [ ]:
import torch
import numpy as np
import random
import os

def set_seed(seed):
    # Python & OS
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

    # NumPy
    np.random.seed(seed)

    # PyTorch (CPU & GPU)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Deterministic CUDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    print(f"Global seed set to: {seed}")

set_seed(SEED)

Global seed set to: 42


# Google Drive Mount

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

FINAL_DIR = Path("/content/drive/MyDrive/Honours Thesis/FINAL")
SOURCE = Path("/content/dataset_complete")

ARCHIVE = FINAL_DIR / "dataset_complete.zip"
assert ARCHIVE.exists(), f"Archive not found: {ARCHIVE}"

!rm -rf /content/dataset_complete /content/temp
!unzip -q "{ARCHIVE}" -d "/content/temp"
!mv /content/temp/ebn_clean /content/dataset_complete
!rm -rf /content/temp

CLASSES = ["0", "1", "2"]
EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

counts = {}
for c in CLASSES:
    counts[c] = sum(
        1 for p in (SOURCE).rglob("*")
        if p.is_file()
        and p.suffix.lower() in EXTS
        and not p.name.startswith(".")
        and any(parent.name == c for parent in p.parents)
    )

print("\nSource dataset")
for c in CLASSES:
    print(f"  grade {c}: {counts[c]:>4} images")
print(f"  total:   {sum(counts.values()):>4} images")

EXPECTED = {"0": 84, "1": 89, "2": 265}
if counts != EXPECTED:
    print(f"\n  [!] Loaded dataset missing image samples")
else:
    print("\n  [OK] Loaded dataset matches the original clean dataset")

Mounted at /content/drive

Source dataset
  grade 0:   84 images
  grade 1:   89 images
  grade 2:  265 images
  total:    438 images

  [OK] Matches the 438 clean originals


In [ ]:
from pathlib import Path

FINAL_DIR = Path("/content/drive/MyDrive/Honours Thesis/FINAL")
GENERATED = Path("/content/dataset_generated")

GEN_ARCHIVE = FINAL_DIR / "dataset_generated.zip"
assert GEN_ARCHIVE.exists(), f"Archive not found: {GEN_ARCHIVE}"

!rm -rf /content/dataset_generated /content/temp_gen
!unzip -q "{GEN_ARCHIVE}" -d "/content/temp_gen"

BUDGETS = [15, 30, 45]
CONFIGS = [
    "baseline_clean",          # no augmentation
    "clahe_regular",           # contrast amplification
    "smallscatter_regular",    # destructive scatter, small
    "mediumscatter_regular",   # destructive scatter, medium
    "largescatter_regular",    # destructive scatter, large
    "remove_regular",          # targeted erasure
]
CLASSES = ["0", "1", "2"]
EXTS    = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

candidates = [
    d for d in Path("/content/temp_gen").rglob("*")
    if d.is_dir() and all((d / str(b)).exists() for b in BUDGETS)
]
assert candidates, "No folder containing 15/30/45 found inside the archive"
root = sorted(candidates, key=lambda p: len(p.parts))[0]

!mv "{root}" /content/dataset_generated
!rm -rf /content/temp_gen

def count_images(d):
    d = Path(d)
    if not d.exists():
        return 0
    return sum(
        1 for p in d.iterdir()
        if p.is_file() and p.suffix.lower() in EXTS and not p.name.startswith(".")
    )


print("\nGenerated counterexamples")
header = f"{'Config':<24}" + "".join(f"{'k=' + str(b):>22}" for b in BUDGETS)
print(header)
print("-" * len(header))

problems = []
for cfg in CONFIGS:
    row = f"{cfg:<24}"
    for b in BUDGETS:
        counts = [count_images(GENERATED / str(b) / cfg / c) for c in CLASSES]
        row += f"{str(counts):>22}"
        if any(n != b for n in counts):
            problems.append(f"{cfg} k={b}: expected {b} per grade, found {counts}")
    print(row)

missing = [
    f"{b}/{cfg}" for b in BUDGETS for cfg in CONFIGS
    if not (GENERATED / str(b) / cfg).exists()
]

print()
if missing:
    print(f"  [!] Missing folders: {missing}")
if problems:
    print("  [!] Count mismatches:")
    for p in problems:
        print(f"      {p}")
if not missing and not problems:
    print(f"  [OK] {len(CONFIGS)} configs x {len(BUDGETS)} budgets, "
          f"{len(CLASSES)}")

print("\nCumulative nesting check")
for cfg in CONFIGS:
    for small, large in [(15, 30), (30, 45)]:
        a = {p.name for c in CLASSES
             for p in (GENERATED / str(small) / cfg / c).glob("*")
             if p.suffix.lower() in EXTS}
        b = {p.name for c in CLASSES
             for p in (GENERATED / str(large) / cfg / c).glob("*")
             if p.suffix.lower() in EXTS}
        if a and b and not a.issubset(b):
            print(f"  [!] {cfg}: k={small} is not a subset of k={large} "
                  f"({len(a - b)} images differ)")
print("  done")


Generated counterexamples
Config                                    k=15                  k=30                  k=45
------------------------------------------------------------------------------------------
baseline_clean                    [15, 15, 15]          [30, 30, 30]          [45, 45, 45]
clahe_regular                     [15, 15, 15]          [30, 30, 30]          [45, 45, 45]
smallscatter_regular              [15, 15, 15]          [30, 30, 30]          [45, 45, 45]
mediumscatter_regular             [15, 15, 15]          [30, 30, 30]          [45, 45, 45]
largescatter_regular              [15, 15, 15]          [30, 30, 30]          [45, 45, 45]
remove_regular                    [15, 15, 15]          [30, 30, 30]          [45, 45, 45]

  [OK] 6 configs x 3 budgets, 3 grades, counts match k per grade

Cumulative nesting check
  done


# Data Splitting and Oversampling

In [ ]:
import re
import hashlib
import shutil
from pathlib import Path
from collections import defaultdict

from PIL import Image

SOURCE  = "/content/dataset_complete"
CV_ROOT = "/content/cv"

TARGET_SIZE = (640, 640)
CLASSES = ["0", "1", "2"]
EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

N_FOLDS = 5
SPLIT_SALT = 42
RATIO_TEST = 0.20
RATIO_LABELED = 0.25
RATIO_TRAIN = 0.70

RATIO_SEED_WITHIN = RATIO_LABELED / (1.0 - RATIO_TEST)
OVERSAMPLE_TRAIN = True
MAX_OVERSAMPLE_RATIO = None
GROUP_KEY = None

def stable_key(text, salt):
    return hashlib.sha256(f"{salt}:{text}".encode("utf-8")).hexdigest()

def is_image(p):
    return (
        p.is_file()
        and p.suffix.lower() in EXTS
        and not p.name.startswith(".")
        and not any(part.startswith(".") for part in p.parts)
    )


def list_images(directory, include_oversample=True):
    directory = Path(directory)
    if not directory.exists():
        return []
    out = [p for p in directory.iterdir() if is_image(p)]
    if not include_oversample:
        out = [p for p in out if not p.name.startswith("oversample_")]
    return sorted(out, key=lambda p: p.name)

def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

def digest_tree(root, content=False):
    root = Path(root)
    entries = []
    for p in sorted(root.rglob("*"), key=lambda p: p.relative_to(root).as_posix()):
        if not is_image(p):
            continue
        rel = p.relative_to(root).as_posix()
        entries.append(f"{rel}:{file_hash(p)}" if content else rel)
    return hashlib.sha256("\n".join(entries).encode("utf-8")).hexdigest()[:16], len(entries)

def discover_images(source, classes):
    source = Path(source)
    found = {c: [] for c in classes}

    for p in source.rglob("*"):
        if not is_image(p):
            continue
        for parent in p.parents:
            if parent.name in classes:
                found[parent.name].append(p)
                break

    for c in classes:
        found[c] = sorted(set(found[c]), key=lambda p: p.relative_to(source).as_posix())

        seen = defaultdict(list)
        for p in found[c]:
            seen[p.name].append(p)
        clashes = {k: v for k, v in seen.items() if len(v) > 1}
        if clashes:
            raise ValueError(
                f"Duplicate filenames in class {c}: {list(clashes)[:5]}. "
                "Rename before splitting -- write-out would overwrite silently."
            )

    return found

def build_units(paths, group_key):
    if group_key is None:
        return [(p.name, [p]) for p in paths]
    buckets = defaultdict(list)
    for p in paths:
        buckets[group_key(p)].append(p)
    return [(gid, sorted(m, key=lambda p: p.name)) for gid, m in sorted(buckets.items())]


def order_units(units, salt):
    units = sorted(units, key=lambda u: stable_key(u[0], salt))
    units.sort(key=lambda u: len(u[1]), reverse=True)
    return units

def make_folds(paths, n_folds, salt, group_key=None):
    units = order_units(build_units(paths, group_key), salt)
    folds = [[] for _ in range(n_folds)]
    for _, members in units:
        target = min(range(n_folds), key=lambda i: (len(folds[i]), i))
        folds[target].extend(members)
    return [sorted(f, key=lambda p: p.name) for f in folds]

def split_remaining(paths, salt, group_key=None):
    units = order_units(build_units(paths, group_key), salt)

    n = sum(len(m) for _, m in units)
    quota = {"seed": int(n * RATIO_SEED_WITHIN)}
    quota["pool"] = n - quota["seed"]
    order = ("seed", "pool")

    bins = {b: [] for b in order}
    for _, members in units:
        target = max(order, key=lambda b: (quota[b] - len(bins[b]), -order.index(b)))
        bins[target].extend(members)

    seed_set = sorted(bins["seed"], key=lambda p: p.name)
    n_train = int(len(seed_set) * RATIO_TRAIN)

    return {
        "train": seed_set[:n_train],
        "val":   seed_set[n_train:],
        "pool":  sorted(bins["pool"], key=lambda p: p.name),
    }

def build_cv_split(by_class, n_folds, salt, group_key=None):
    per_class_folds = {
        c: make_folds(by_class[c], n_folds, salt + int(c), group_key) for c in CLASSES
    }

    cv = []
    for k in range(n_folds):
        fold = {}
        for c in CLASSES:
            test = per_class_folds[c][k]
            rest = sorted(
                (p for j, f in enumerate(per_class_folds[c]) if j != k for p in f),
                key=lambda p: p.name,
            )
            sub = split_remaining(rest, salt + 1000 * k + int(c), group_key)
            fold[c] = {"test": test, **sub}
        cv.append(fold)
    return cv

def resize_and_save(src, dst, size):
    with Image.open(src) as img:
        out = img.convert("RGB").resize(size, Image.Resampling.LANCZOS)
        if Path(dst).suffix.lower() in (".jpg", ".jpeg"):
            out.save(dst, format="JPEG", quality=95, subsampling=0,
                     optimize=False, progressive=False)
        else:
            out.save(dst)

def build_fold_datasets(fold, fold_dir):
    fold_dir = Path(fold_dir)
    partial  = fold_dir / "dataset_train_partial"
    complete = fold_dir / "dataset_train_complete"
    unlab    = fold_dir / "unlabeled"

    if fold_dir.exists():
        shutil.rmtree(fold_dir)

    for part in ("train", "val", "test"):
        for c in CLASSES:
            (partial / part / c).mkdir(parents=True, exist_ok=True)

    for c in CLASSES:
        for part in ("train", "val", "test"):
            for src in fold[c][part]:
                resize_and_save(src, partial / part / c / src.name, TARGET_SIZE)

    complete.mkdir(parents=True, exist_ok=True)
    shutil.copytree(partial / "val",   complete / "val")
    shutil.copytree(partial / "test",  complete / "test")
    shutil.copytree(partial / "train", complete / "train")

    for c in CLASSES:
        for src in fold[c]["pool"]:
            resize_and_save(src, complete / "train" / c / src.name, TARGET_SIZE)

    unlab.mkdir(parents=True, exist_ok=True)
    for c in CLASSES:
        for src in fold[c]["pool"]:
            resize_and_save(src, unlab / f"({c})_{src.name}", TARGET_SIZE)

    return partial, complete, unlab

def oversample_train(root, salt, max_ratio=None, label=""):
    train_dir = Path(root) / "train"

    removed = 0
    for c in CLASSES:
        for p in sorted(train_dir.joinpath(c).glob("oversample_*"), key=lambda p: p.name):
            p.unlink()
            removed += 1
    if removed:
        print(f"      cleared {removed} duplicates from a previous run")

    files = {c: list_images(train_dir / c, include_oversample=False) for c in CLASSES}
    target = max(len(v) for v in files.values())

    parts = []
    for c in CLASSES:
        originals = files[c]
        have = len(originals)
        if have == 0 or have >= target:
            parts.append(f"{c}: {have}")
            continue

        cap = target if max_ratio is None else min(target, int(have * max_ratio))
        deficit = cap - have

        to_copy = []
        for rep in range(deficit // have):
            to_copy += [(p, f"r{rep}") for p in originals]

        rem = deficit % have
        if rem:
            ordered = sorted(originals, key=lambda p: stable_key(p.name, salt + int(c)))
            to_copy += [(p, "rem") for p in ordered[:rem]]

        for src, tag in to_copy:
            shutil.copy2(src, train_dir / c / f"oversample_{tag}_{src.name}")

        parts.append(f"{c}: {have}->{have + len(to_copy)}")

    print(f"      {label} train balanced to {target}/class  ({', '.join(parts)})")

def base_name(name):
    name = re.sub(r"^oversample_(?:r\d+|rem)_", "", name)
    return re.sub(r"^\(\d\)_", "", name)

def names_in(root, part):
    return {base_name(p.name) for c in CLASSES for p in list_images(Path(root) / part / c)}

def verify_fold(partial, complete, unlab, group_key=None):
    ok = True

    for part in ("val", "test"):
        a = {(c, p.name): file_hash(p)
             for c in CLASSES for p in list_images(Path(partial) / part / c)}
        b = {(c, p.name): file_hash(p)
             for c in CLASSES for p in list_images(Path(complete) / part / c)}
        if a != b:
            ok = False
            print(f"      [X] {part} differs between partial and complete")

    pool_names = {base_name(p.name) for p in list_images(unlab)}
    for label, root in (("partial", partial), ("complete", complete)):
        parts = {p: names_in(root, p) for p in ("train", "val", "test")}
        parts["pool"] = pool_names
        keys = ["train", "val", "test", "pool"]
        for i, x in enumerate(keys):
            for y in keys[i + 1:]:
                if label == "complete" and {x, y} == {"train", "pool"}:
                    continue   # complete/train contains the pool by design
                overlap = parts[x] & parts[y]
                if overlap:
                    ok = False
                    print(f"      [X] {label}: {x}/{y} share {len(overlap)}: "
                          f"{sorted(overlap)[:3]}")

    if group_key is not None:
        for label, root in (("partial", partial), ("complete", complete)):
            g = {p: {group_key(n) for n in names_in(root, p)}
                 for p in ("train", "val", "test")}
            for x, y in (("train", "val"), ("train", "test"), ("val", "test")):
                shared = g[x] & g[y]
                if shared:
                    ok = False
                    print(f"      [X] {label}: {x}/{y} share groups {sorted(shared)[:3]}")

    return ok

def verify_folds_cover_dataset(cv, by_class):
    ok = True
    for c in CLASSES:
        seen = []
        for fold in cv:
            seen += [p.name for p in fold[c]["test"]]
        expected = {p.name for p in by_class[c]}
        if len(seen) != len(set(seen)):
            ok = False
            print(f"  [X] grade {c}: an image appears in more than one test fold")
        if set(seen) != expected:
            ok = False
            missing = expected - set(seen)
            print(f"  [X] grade {c}: {len(missing)} images never tested")
    return ok

by_class = discover_images(SOURCE, CLASSES)

print("=" * 74)
print("SOURCE DATASET")
print("=" * 74)
for c in CLASSES:
    print(f"  grade {c}: {len(by_class[c])} images")
print(f"  total:   {sum(len(v) for v in by_class.values())} images")

if GROUP_KEY is not None:
    for c in CLASSES:
        print(f"  grade {c}: {len({GROUP_KEY(p) for p in by_class[c]})} groups")

cv = build_cv_split(by_class, N_FOLDS, SPLIT_SALT, GROUP_KEY)

print("\n" + "=" * 74)
print(f"{N_FOLDS}-FOLD SPLIT  (on originals, before any oversampling)")
print("=" * 74)
print(f"{'Fold':<6}{'Partition':<18}" + "".join(f"{'Grade ' + c:>9}" for c in CLASSES)
      + f"{'Total':>9}")
print("-" * 74)
for k, fold in enumerate(cv):
    for part, label in [("test", "Test (held out)"), ("train", "Seed train"),
                        ("val", "Val (shared)"), ("pool", "Annotation pool")]:
        counts = [len(fold[c][part]) for c in CLASSES]
        print(f"{('' if part != 'test' else str(k)):<6}{label:<18}"
              + "".join(f"{n:>9}" for n in counts) + f"{sum(counts):>9}")
    print("-" * 74)

if verify_folds_cover_dataset(cv, by_class):
    print("  [OK] every image is tested exactly once across the folds")

print("\n" + "=" * 74)
print("BUILDING FOLD DATASETS")
print("=" * 74)

CV_ROOT = Path(CV_ROOT)
FOLD_DIRS = {}
all_ok = True

for k, fold in enumerate(cv):
    fold_dir = CV_ROOT / f"fold{k}"
    print(f"\n  fold {k}")
    partial, complete, unlab = build_fold_datasets(fold, fold_dir)

    if OVERSAMPLE_TRAIN:
        oversample_train(partial,  SPLIT_SALT + k, MAX_OVERSAMPLE_RATIO, "partial ")
        oversample_train(complete, SPLIT_SALT + k, MAX_OVERSAMPLE_RATIO, "complete")

    ok = verify_fold(partial, complete, unlab,
                     group_key=(lambda n: re.match(r'^(\d+)-', n).group(1))
                     if GROUP_KEY else None)
    all_ok &= ok
    print(f"      verification: {'clean' if ok else 'PROBLEM FOUND'}")

    FOLD_DIRS[k] = {"partial": str(partial), "complete": str(complete),
                    "unlabeled": str(unlab)}

print("\n" + "=" * 74)
print("FINAL FOLD DATASETS")
print("=" * 74)
for k in range(N_FOLDS):
    print(f"\n  fold {k}")
    for label in ("partial", "complete"):
        root = Path(FOLD_DIRS[k][label])
        sizes = {part: sum(len(list_images(root / part / c)) for c in CLASSES)
                 for part in ("train", "val", "test")}
        print(f"    {label:<9} train {sizes['train']:>4}   "
              f"val {sizes['val']:>3}   test {sizes['test']:>3}")
    pool = len(list_images(Path(FOLD_DIRS[k]["unlabeled"])))
    print(f"    {'unlabeled':<9} pool  {pool:>4}")

print("\n" + "=" * 74)
print("FINGERPRINTS  (identical across runs and runtimes)")
print("=" * 74)
for k in range(N_FOLDS):
    d, n = digest_tree(CV_ROOT / f"fold{k}", content=False)
    print(f"  fold {k}: names {d}  ({n} files)")
d_all, n_all = digest_tree(CV_ROOT, content=False)
print(f"\n  all folds: names {d_all}  ({n_all} files)  split salt {SPLIT_SALT}")

print("\n" + ("VERDICT: no leakage detected." if all_ok else "VERDICT: PROBLEM FOUND."))
print("=" * 74)

SOURCE DATASET
  grade 0: 84 images
  grade 1: 89 images
  grade 2: 265 images
  total:   438 images

5-FOLD SPLIT  (on originals, before any oversampling)
Fold  Partition           Grade 0  Grade 1  Grade 2    Total
--------------------------------------------------------------------------
0     Test (held out)          17       18       53       88
      Seed train               14       15       46       75
      Val (shared)              6        7       20       33
      Annotation pool          47       49      146      242
--------------------------------------------------------------------------
1     Test (held out)          17       18       53       88
      Seed train               14       15       46       75
      Val (shared)              6        7       20       33
      Annotation pool          47       49      146      242
--------------------------------------------------------------------------
2     Test (held out)          17       18       53       88
      See

# Helper Functions

In [ ]:
import os
import json
import random
import hashlib
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm
from torchvision import transforms
from ultralytics import YOLO
from sklearn.metrics import (
    cohen_kappa_score,
    precision_recall_fscore_support,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix,
)

CLASSES = ["0", "1", "2"]
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to: {seed}")

def list_images(directory):
    directory = Path(directory)
    if not directory.exists():
        return []
    return sorted(
        (p for p in directory.iterdir()
         if p.is_file() and p.suffix.lower() in IMAGE_EXTS and not p.name.startswith(".")),
        key=lambda p: p.name,
    )

def get_masks(test_dir, cache_path=None, force=False):
    from rembg import remove, new_session

    test_dir = Path(test_dir)
    image_paths = [p for c in CLASSES for p in list_images(test_dir / c)]

    if not image_paths:
        raise FileNotFoundError(f"No images under {test_dir}")

    if cache_path is not None:
        cache_path = Path(cache_path)
        if cache_path.exists() and not force:
            cached = torch.load(cache_path, weights_only=False)
            if set(cached.keys()) == {str(p) for p in image_paths}:
                print(f"  Loaded {len(cached)} cached masks from {cache_path.name}")
                return cached
            print("  Cache does not match this test set -- regenerating")

    session = new_session("u2net")
    masks = {}

    for img_path in tqdm(image_paths, desc="  Generating masks", leave=False):
        img = Image.open(img_path).convert("RGB")
        w, h = img.size
        mask = np.array(remove(img, session=session, only_mask=True))
        binary = (mask > 128).astype(np.uint8)
        if binary.shape != (h, w):
            binary = cv2.resize(binary, (w, h), interpolation=cv2.INTER_NEAREST)
        masks[str(img_path)] = binary

    if cache_path is not None:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(masks, cache_path)
        print(f"  Cached {len(masks)} masks to {cache_path.name}")

    return masks

def _compute_gradcam(torch_model, input_tensor):
    activations, gradients = {}, {}
    target_layer = torch_model.model[-2]

    fh = target_layer.register_forward_hook(
        lambda m, i, o: activations.__setitem__("v", o.clone()))
    bh = target_layer.register_full_backward_hook(
        lambda m, gi, go: gradients.__setitem__("v", go[0]))

    try:
        torch_model.zero_grad()
        out = torch_model(input_tensor)
        if isinstance(out, (list, tuple)):
            out = out[0]
        pred = out.argmax(dim=1).item()
        out[0, pred].backward()

        weights = gradients["v"].mean(dim=[2, 3], keepdim=True)
        cam = F.relu((weights * activations["v"]).sum(dim=1, keepdim=True)).squeeze()

        lo, hi = cam.min(), cam.max()
        cam = (cam - lo) / (hi - lo) if (hi - lo) > 1e-8 else torch.zeros_like(cam)
        return cam.detach().cpu().numpy().astype(np.float32)
    finally:
        fh.remove()
        bh.remove()

def saliency_metrics(masks, model, imgsz=640, tau_percentile=25.0):
    if not masks:
        return {"FFP": None, "BFP": None, "BSR": None,
                "FFP_std": None, "BFP_std": None, "BSR_std": None}

    device = "cuda" if torch.cuda.is_available() else "cpu"
    net = model.model
    net.eval().to(device)

    preprocess = transforms.Compose([
        transforms.Resize((imgsz, imgsz)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    ffp, bfp, bsr = [], [], []

    for img_path, mask in tqdm(sorted(masks.items()), desc="  GradCAM", leave=False):
        p = Path(img_path)
        if not p.exists():
            continue
        try:
            img = Image.open(p).convert("RGB")
            w, h = img.size

            x = preprocess(img).unsqueeze(0).to(device).clone().detach()
            x.requires_grad_(True)

            cam = cv2.resize(_compute_gradcam(net, x), (w, h), interpolation=cv2.INTER_LINEAR)

            m = mask
            if m.shape != (h, w):
                m = cv2.resize(m.astype(np.float32), (w, h),
                               interpolation=cv2.INTER_NEAREST).astype(np.uint8)

            fg, bg = (m == 1), (m == 0)
            tau = np.percentile(cam, tau_percentile)
            total = cam.sum()

            ffp.append(float(((cam > tau) & fg).sum() / fg.sum()) if fg.sum() else 0.0)
            bfp.append(float(((cam > tau) & bg).sum() / bg.sum()) if bg.sum() else 0.0)
            bsr.append(float((cam * (1 - m)).sum() / total) if total > 1e-8 else 0.0)
        except Exception as e:
            print(f"    [saliency] skipped {p.name}: {e}")

    if not ffp:
        return {"FFP": None, "BFP": None, "BSR": None,
                "FFP_std": None, "BFP_std": None, "BSR_std": None}

    return {
        "FFP": float(np.mean(ffp)), "FFP_std": float(np.std(ffp)),
        "BFP": float(np.mean(bfp)), "BFP_std": float(np.std(bfp)),
        "BSR": float(np.mean(bsr)), "BSR_std": float(np.std(bsr)),
    }

def collect_predictions(model, test_root):
    test_root = Path(test_root)
    name_to_idx = {str(v): k for k, v in model.names.items()}

    y_true, y_pred = [], []
    for c in CLASSES:
        true_label = name_to_idx.get(c, int(c) if c.isdigit() else None)
        if true_label is None:
            print(f"    [warn] class folder {c} not in model.names -- skipped")
            continue
        for img_path in list_images(test_root / c):
            r = model(str(img_path), verbose=False)
            y_true.append(true_label)
            y_pred.append(int(r[0].probs.top1))

    return np.array(y_true), np.array(y_pred)

def classification_metrics(y_true, y_pred):
    if len(y_true) == 0:
        return {}

    out = {
        "accuracy": float((y_true == y_pred).mean()),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "QWK": float(cohen_kappa_score(y_true, y_pred, weights="quadratic")),
        "kappa_linear": float(cohen_kappa_score(y_true, y_pred, weights="linear")),
        "MCC": float(matthews_corrcoef(y_true, y_pred)),
    }

    for avg in ("macro", "weighted"):
        p, r, f, _ = precision_recall_fscore_support(
            y_true, y_pred, average=avg, zero_division=0)
        out[f"precision_{avg}"] = float(p)
        out[f"recall_{avg}"] = float(r)
        out[f"f1_{avg}"] = float(f)

    labels = list(range(len(CLASSES)))
    p, r, f, s = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0)
    for i, c in enumerate(CLASSES):
        out[f"precision_grade{c}"] = float(p[i])
        out[f"recall_grade{c}"] = float(r[i])
        out[f"f1_grade{c}"] = float(f[i])
        out[f"support_grade{c}"] = int(s[i])

    out["confusion_matrix"] = json.dumps(
        confusion_matrix(y_true, y_pred, labels=labels).tolist())
    return out

def evaluate_ebn_model(model_path, dataset_path, masks,
                       split="test", imgsz=640, tau_percentile=25.0, verbose=True):
    model = YOLO(model_path, task="classify")
    test_root = Path(dataset_path) / split

    y_true, y_pred = collect_predictions(model, test_root)
    metrics = classification_metrics(y_true, y_pred)
    metrics["n_test"] = int(len(y_true))

    saliency_model = YOLO(model_path, task="classify")
    with torch.set_grad_enabled(True):
        saliency_model.model.eval()
        metrics.update(saliency_metrics(
            masks, saliency_model, imgsz=imgsz, tau_percentile=tau_percentile))

    if verbose:
        print(f"    acc={metrics['accuracy']:.4f}  QWK={metrics['QWK']:.4f}  "
              f"macroF1={metrics['f1_macro']:.4f}  FFP={metrics['FFP']:.4f}  "
              f"BFP={metrics['BFP']:.4f}  BSR={metrics['BSR']:.4f}")

    return metrics

print("Metrics helpers loaded.")

Metrics helpers loaded.


# Model Training

In [ ]:
import os
import time
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from ultralytics import YOLO

FINAL_DIR   = Path("/content/drive/MyDrive/Honours Thesis/FINAL")
RUNS_DIR    = FINAL_DIR / "train_runs"
RESULTS_DIR = FINAL_DIR / "train_results"
CACHE_DIR   = FINAL_DIR / "cache"
LOCAL_RUNS  = Path("/content/runs_local")

for d in (RUNS_DIR, RESULTS_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

N_FOLDS   = 5
CV_ROOT   = Path("/content/cv")
DATASETS  = ["partial", "complete"]
SEEDS     = [42, 1337, 314, 0, 123]

BASE_WEIGHTS = "yolo11s-cls.pt"
EPOCHS       = 50
PATIENCE     = 15
IMGSZ        = 640
BATCH        = 16
WORKERS      = 0
TAU_PCT      = 25.0

RAW_CSV      = RESULTS_DIR / "cv_raw.csv"
SUMMARY_CSV  = RESULTS_DIR / "cv_summary.csv"
BYFOLD_CSV   = RESULTS_DIR / "cv_by_fold.csv"
VARIANCE_CSV = RESULTS_DIR / "cv_variance.csv"
PIVOT_CSV    = RESULTS_DIR / "cv_report.csv"

METRIC_COLS = [
    "accuracy", "balanced_accuracy", "QWK", "kappa_linear", "MCC",
    "precision_macro", "recall_macro", "f1_macro",
    "precision_weighted", "recall_weighted", "f1_weighted",
    "f1_grade0", "f1_grade1", "f1_grade2",
    "recall_grade0", "recall_grade1", "recall_grade2",
    "precision_grade0", "precision_grade1", "precision_grade2",
    "FFP", "BFP", "BSR",
    "epochs_run", "best_epoch", "train_minutes",
]

def dataset_path(fold, dataset):
    return CV_ROOT / f"fold{fold}" / f"dataset_train_{dataset}"

def remap_masks(masks, from_root, to_root):
    from_root, to_root = Path(from_root), Path(to_root)
    out = {}
    for k, v in masks.items():
        rel = Path(k).relative_to(from_root)
        out[str(to_root / rel)] = v
    return out

print("=" * 74)
print("PREPARING FOREGROUND MASKS")
print("=" * 74)

MASKS = {}
for fold in range(N_FOLDS):
    p_root = dataset_path(fold, "partial")
    c_root = dataset_path(fold, "complete")
    print(f"  fold {fold}")
    m = get_masks(p_root / "test", cache_path=CACHE_DIR / f"masks_fold{fold}.pt")
    MASKS[(fold, "partial")] = m
    MASKS[(fold, "complete")] = remap_masks(m, p_root / "test", c_root / "test")

if RAW_CSV.exists():
    master_df = pd.read_csv(RAW_CSV)
    records = master_df.to_dict("records")
    print(f"\nResuming: {len(records)} of {len(DATASETS) * N_FOLDS * len(SEEDS)} runs recorded")
else:
    master_df = pd.DataFrame()
    records = []

def already_done(dataset, fold, seed):
    if master_df.empty:
        return False
    hit = master_df[
        (master_df["dataset"] == dataset)
        & (master_df["fold"] == fold)
        & (master_df["seed"] == seed)
    ]
    return not hit.empty

def read_training_curve(run_dir):
    csv_path = Path(run_dir) / "results.csv"
    if not csv_path.exists():
        return None, None
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    acc_col = next((c for c in df.columns if "accuracy_top1" in c), None)
    best = int(df[acc_col].idxmax()) + 1 if acc_col else None
    return len(df), best

total = len(DATASETS) * N_FOLDS * len(SEEDS)
done = 0

for dataset in DATASETS:
    for fold in range(N_FOLDS):
        for seed in SEEDS:
            done += 1

            if already_done(dataset, fold, seed):
                print(f"[skip {done}/{total}] {dataset} | fold {fold} | seed {seed}")
                continue

            print("\n" + "=" * 74)
            print(f"[{done}/{total}]  dataset={dataset}  fold={fold}  seed={seed}")
            print("=" * 74)

            set_seed(seed)

            data_dir = dataset_path(fold, dataset)
            run_name = f"{dataset}_fold{fold}_seed{seed}"

            for cache in Path(data_dir).rglob("*.cache"):
                cache.unlink()

            local_dir = LOCAL_RUNS / run_name
            if local_dir.exists():
                shutil.rmtree(local_dir)

            model = YOLO(BASE_WEIGHTS)

            t0 = time.time()
            model.train(
                data=str(data_dir),
                project=str(LOCAL_RUNS),
                name=run_name,
                exist_ok=True,
                epochs=EPOCHS,
                patience=PATIENCE,
                imgsz=IMGSZ,
                batch=BATCH,
                workers=WORKERS,
                seed=seed,
                deterministic=True,
                erasing=0.0,
                fliplr=0.5,
                flipud=0.5,
                degrees=15.0,
                hsv_h=0.028,
                hsv_s=0.2,
                hsv_v=0.2,
            )
            train_minutes = (time.time() - t0) / 60.0

            best_ckpt = local_dir / "weights" / "best.pt"
            metrics = evaluate_ebn_model(
                model_path=str(best_ckpt),
                dataset_path=str(data_dir),
                masks=MASKS[(fold, dataset)],
                split="test",
                imgsz=IMGSZ,
                tau_percentile=TAU_PCT,
            )

            epochs_run, best_epoch = read_training_curve(local_dir)

            drive_dir = RUNS_DIR / run_name
            if drive_dir.exists():
                shutil.rmtree(drive_dir)
            drive_dir.mkdir(parents=True, exist_ok=True)
            (drive_dir / "weights").mkdir(exist_ok=True)
            shutil.copy2(best_ckpt, drive_dir / "weights" / "best.pt")
            for extra in ("results.csv", "args.yaml"):
                src = local_dir / extra
                if src.exists():
                    shutil.copy2(src, drive_dir / extra)

            records.append({
                "dataset": dataset,
                "fold": fold,
                "seed": seed,
                "run_name": run_name,
                "weights": str(drive_dir / "weights" / "best.pt"),
                "epochs_run": epochs_run,
                "best_epoch": best_epoch,
                "train_minutes": round(train_minutes, 2),
                **metrics,
            })

            master_df = pd.DataFrame(records)
            master_df.to_csv(RAW_CSV, index=False)
            print(f"  saved -> {RAW_CSV.name}  ({len(records)}/{total} runs)")

            shutil.rmtree(local_dir, ignore_errors=True)

print("\n" + "=" * 74)
print("BUILDING SUMMARY")
print("=" * 74)

master_df = pd.read_csv(RAW_CSV)
present = [c for c in METRIC_COLS if c in master_df.columns]

def describe(vals):
    n = len(vals)
    mean = float(np.mean(vals))
    sd = float(np.std(vals, ddof=1)) if n > 1 else 0.0
    sem = sd / np.sqrt(n) if n > 1 else 0.0
    ci = sem * stats.t.ppf(0.975, n - 1) if n > 1 else 0.0
    return {
        "n": n, "mean": mean, "std": sd, "sem": sem,
        "ci95_low": mean - ci, "ci95_high": mean + ci,
        "min": float(np.min(vals)), "q1": float(np.percentile(vals, 25)),
        "median": float(np.median(vals)), "q3": float(np.percentile(vals, 75)),
        "max": float(np.max(vals)), "range": float(np.max(vals) - np.min(vals)),
        "cv_pct": (sd / mean * 100) if mean else np.nan,
    }

summary_rows = []
for dataset, group in master_df.groupby("dataset"):
    for metric in present:
        vals = pd.to_numeric(group[metric], errors="coerce").dropna().values
        if len(vals):
            summary_rows.append({"dataset": dataset, "metric": metric, **describe(vals)})
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_CSV, index=False)

byfold_rows = []
for (dataset, fold), group in master_df.groupby(["dataset", "fold"]):
    for metric in present:
        vals = pd.to_numeric(group[metric], errors="coerce").dropna().values
        if len(vals):
            byfold_rows.append({"dataset": dataset, "fold": fold, "metric": metric,
                                **describe(vals)})
byfold_df = pd.DataFrame(byfold_rows)
byfold_df.to_csv(BYFOLD_CSV, index=False)

var_rows = []
for dataset, group in master_df.groupby("dataset"):
    for metric in present:
        vals = pd.to_numeric(group[metric], errors="coerce")
        sub = group.assign(_v=vals).dropna(subset=["_v"])
        if sub.empty:
            continue
        fold_means = sub.groupby("fold")["_v"].mean().values
        within = sub.groupby("fold")["_v"].std(ddof=1).dropna().values
        var_rows.append({
            "dataset": dataset,
            "metric": metric,
            "grand_mean": float(sub["_v"].mean()),
            "between_fold_std": float(np.std(fold_means, ddof=1)) if len(fold_means) > 1 else 0.0,
            "within_fold_std": float(np.sqrt(np.mean(within ** 2))) if len(within) else 0.0,
            "fold_min": float(np.min(fold_means)),
            "fold_max": float(np.max(fold_means)),
        })
variance_df = pd.DataFrame(var_rows)
variance_df["split_dominates"] = (
    variance_df["between_fold_std"] > variance_df["within_fold_std"])
variance_df.to_csv(VARIANCE_CSV, index=False)

pivot = summary_df.copy()
pivot["value"] = pivot.apply(lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1)
pivot = pivot.pivot(index="metric", columns="dataset", values="value")
pivot = pivot.reindex([m for m in METRIC_COLS if m in pivot.index])
pivot.to_csv(PIVOT_CSV)

print(f"\n  raw       -> {RAW_CSV.name}   (one row per run)")
print(f"  summary   -> {SUMMARY_CSV.name}")
print(f"  by fold   -> {BYFOLD_CSV.name}")
print(f"  variance  -> {VARIANCE_CSV.name}")
print(f"  report    -> {PIVOT_CSV.name}")

headline = ["accuracy", "balanced_accuracy", "QWK", "f1_macro", "FFP", "BFP", "BSR"]

print("\n" + "=" * 74)
print(f"CROSS-VALIDATED RESULTS  (mean ± std over {N_FOLDS} folds x {len(SEEDS)} seeds)")
print("=" * 74)
print(f"{'Metric':<20}" + "".join(f"{d:>26}" for d in DATASETS))
print("-" * 74)
for metric in headline:
    row = f"{metric:<20}"
    for d in DATASETS:
        s = summary_df[(summary_df["dataset"] == d) & (summary_df["metric"] == metric)]
        if s.empty:
            row += f"{'n/a':>26}"
        else:
            row += (f"{s['mean'].iat[0]:>13.4f} ± {s['std'].iat[0]:.4f}"
                    f"  [{s['ci95_low'].iat[0]:.3f},{s['ci95_high'].iat[0]:.3f}]")
    print(row)

print("\n" + "=" * 74)
print("VARIANCE SOURCES  (which matters more: the split, or training noise?)")
print("=" * 74)
print(f"{'Dataset':<10}{'Metric':<20}{'between-fold':>14}{'within-fold':>14}{'dominant':>14}")
print("-" * 74)
for d in DATASETS:
    for metric in headline:
        v = variance_df[(variance_df["dataset"] == d) & (variance_df["metric"] == metric)]
        if v.empty:
            continue
        b, w = v["between_fold_std"].iat[0], v["within_fold_std"].iat[0]
        print(f"{d:<10}{metric:<20}{b:>14.4f}{w:>14.4f}"
              f"{('split' if b > w else 'seed'):>14}")
print("=" * 74)

PREPARING FOREGROUND MASKS
  fold 0
  Loaded 88 cached masks from masks_fold0.pt
  fold 1
  Loaded 88 cached masks from masks_fold1.pt
  fold 2
  Loaded 88 cached masks from masks_fold2.pt
  fold 3
  Loaded 88 cached masks from masks_fold3.pt
  fold 4
  Loaded 86 cached masks from masks_fold4.pt

Resuming: 48 of 50 runs recorded
[skip 1/50] partial | fold 0 | seed 42
[skip 2/50] partial | fold 0 | seed 1337
[skip 3/50] partial | fold 0 | seed 314
[skip 4/50] partial | fold 0 | seed 0
[skip 5/50] partial | fold 0 | seed 123
[skip 6/50] partial | fold 1 | seed 42
[skip 7/50] partial | fold 1 | seed 1337
[skip 8/50] partial | fold 1 | seed 314
[skip 9/50] partial | fold 1 | seed 0
[skip 10/50] partial | fold 1 | seed 123
[skip 11/50] partial | fold 2 | seed 42
[skip 12/50] partial | fold 2 | seed 1337
[skip 13/50] partial | fold 2 | seed 314
[skip 14/50] partial | fold 2 | seed 0
[skip 15/50] partial | fold 2 | seed 123
[skip 16/50] partial | fold 3 | seed 42
[skip 17/50] partial | fold 3

    acc=0.8256  QWK=0.8135  macroF1=0.7785  FFP=0.6143  BFP=0.6506  BSR=0.5537
  saved -> cv_raw.csv  (49/50 runs)

[50/50]  dataset=complete  fold=4  seed=123
Global seed set to: 123
Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cv/fold4/dataset_train_complete, degrees=15.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.0, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.028, hsv_s=0.2, hsv_v=0.2, imgsz=640, iou=0.7, keras=False, ko

    acc=0.6744  QWK=0.6226  macroF1=0.6307  FFP=0.5559  BFP=0.5209  BSR=0.4828
  saved -> cv_raw.csv  (50/50 runs)

BUILDING SUMMARY

  raw       -> cv_raw.csv   (one row per run)
  summary   -> cv_summary.csv
  by fold   -> cv_by_fold.csv
  variance  -> cv_variance.csv
  report    -> cv_report.csv

CROSS-VALIDATED RESULTS  (mean ± std over 5 folds x 5 seeds)
Metric                                 partial                  complete
--------------------------------------------------------------------------
accuracy                   0.6353 ± 0.0641  [0.609,0.662]       0.7356 ± 0.0711  [0.706,0.765]
balanced_accuracy          0.4818 ± 0.0956  [0.442,0.521]       0.6681 ± 0.0968  [0.628,0.708]
QWK                        0.3496 ± 0.1845  [0.273,0.426]       0.6568 ± 0.1121  [0.611,0.703]
f1_macro                   0.4538 ± 0.1118  [0.408,0.500]       0.6575 ± 0.0996  [0.616,0.699]
FFP                        0.6693 ± 0.1525  [0.606,0.732]       0.5044 ± 0.1657  [0.436,0.573]
BFP            

# Model Finetuning


In [ ]:
import os
import re
import time
import shutil
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from ultralytics import YOLO

FINAL_DIR    = Path("/content/drive/MyDrive/Honours Thesis/FINAL")
TRAIN_RUNS   = FINAL_DIR / "train_runs"
RUNS_DIR     = FINAL_DIR / "finetune_runs"
RESULTS_DIR  = FINAL_DIR / "finetune_results"
CACHE_DIR    = FINAL_DIR / "cache"
LOCAL_RUNS   = Path("/content/finetune_local")

for d in (RUNS_DIR, RESULTS_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

FOLD       = 2
CV_ROOT    = Path("/content/cv")
DATA_SRC   = CV_ROOT / f"fold{FOLD}" / "dataset_train_partial"
GENERATED  = Path("/content/dataset_generated")
WORK_DIR   = Path("/content/dataset_finetune")

BASE_CKPT = TRAIN_RUNS / f"partial_fold{FOLD}_seed0" / "weights" / "best.pt"
assert BASE_CKPT.exists(), f"Baseline checkpoint not found: {BASE_CKPT}"

CONFIGS = [
    "baseline_clean",
    "clahe_regular",
    "smallscatter_regular",
    "mediumscatter_regular",
    "largescatter_regular",
    "remove_regular",
]
CONTROL = "baseline_clean"
BUDGETS = [15, 30, 45]
SEEDS = [42, 1337, 314, 0, 123]

EPOCHS = 30
PATIENCE = 0
IMGSZ = 640
BATCH = 8
FREEZE = 9
OPTIMIZER = "AdamW"
LR0 = 5e-4
LRF = 0.05
COS_LR = True
WEIGHT_DECAY = 5e-4
WARMUP_EPOCHS = 1.0
WARMUP_BIAS_LR = 0.0
WARMUP_MOMENTUM = 0.8
DROPOUT = 0.15
WORKERS = 8
TAU_PCT = 25.0

EVAL_BEST_TOO = True
SAVE_WEIGHTS  = False

RAW_CSV = RESULTS_DIR / "finetune_raw.csv"
SUMMARY_CSV = RESULTS_DIR / "finetune_summary.csv"
DELTA_CSV = RESULTS_DIR / "finetune_deltas.csv"
BYCONF_CSV = RESULTS_DIR / "finetune_by_config.csv"
PIVOT_CSV = RESULTS_DIR / "finetune_report.csv"
REF_CSV = RESULTS_DIR / "finetune_reference.csv"

METRIC_COLS = [
    "accuracy", "balanced_accuracy", "QWK", "kappa_linear", "MCC",
    "precision_macro", "recall_macro", "f1_macro",
    "precision_weighted", "recall_weighted", "f1_weighted",
    "f1_grade0", "f1_grade1", "f1_grade2",
    "recall_grade0", "recall_grade1", "recall_grade2",
    "precision_grade0", "precision_grade1", "precision_grade2",
    "FFP", "BFP", "BSR",
    "epochs_run", "best_epoch", "train_minutes",
]

DELTA_COLS = ["config", "budget", "metric", "n_pairs", "mean_delta",
              "ci95_low", "ci95_high", "p_value", "cohen_d", "significant_at_05"]
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
CLASSES    = ["0", "1", "2"]

def imgs_in(directory):
    directory = Path(directory)
    if not directory.exists():
        return []
    return sorted(
        (p for p in directory.iterdir()
         if p.is_file() and p.suffix.lower() in IMAGE_EXTS
         and not p.name.startswith(".")),
        key=lambda p: p.name,
    )

def source_name(name):
    name = re.sub(r"^gen_[a-z_]+_", "", name)
    name = re.sub(r"^oversample_(?:r\d+|rem)_", "", name)
    name = re.sub(r"^\(\d\)_", "", name)
    return name

def digest_tree(root):
    root = Path(root)
    names = sorted(
        p.relative_to(root).as_posix() for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
        and not p.name.startswith(".")
    )
    return hashlib.sha256("\n".join(names).encode("utf-8")).hexdigest()[:16], len(names)

def prepare_finetune_dataset(config, budget, verbose=False):
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)

    for part in ("train", "val"):
        for c in CLASSES:
            (WORK_DIR / part / c).mkdir(parents=True, exist_ok=True)
            for src in imgs_in(DATA_SRC / part / c):
                shutil.copy2(src, WORK_DIR / part / c / src.name)

    added = {}
    for c in CLASSES:
        files = imgs_in(GENERATED / str(budget) / config / c)
        if len(files) != budget:
            raise ValueError(
                f"{config} k={budget} grade {c}: expected {budget}, found {len(files)}")
        for src in files:
            shutil.copy2(src, WORK_DIR / "train" / c / f"gen_{config}_{src.name}")
        added[c] = len(files)

    for cache in WORK_DIR.rglob("*.cache"):
        cache.unlink()

    if verbose:
        counts = [len(imgs_in(WORK_DIR / "train" / c)) for c in CLASSES]
        print(f"      train {sum(counts)} ({counts}), added {list(added.values())} per grade")

    return WORK_DIR

def verify_no_leakage(work_dir):
    train = {source_name(p.name) for c in CLASSES for p in imgs_in(Path(work_dir) / "train" / c)}
    val   = {source_name(p.name) for c in CLASSES for p in imgs_in(Path(work_dir) / "val" / c)}
    test  = {source_name(p.name) for c in CLASSES for p in imgs_in(DATA_SRC / "test" / c)}

    bad = []
    for name, (a, b) in {"train/val": (train, val), "train/test": (train, test),
                         "val/test": (val, test)}.items():
        if a & b:
            bad.append(f"{name} share {len(a & b)}: {sorted(a & b)[:3]}")
    if bad:
        raise RuntimeError("LEAKAGE: " + "; ".join(bad))

def read_training_curve(run_dir):
    csv_path = Path(run_dir) / "results.csv"
    if not csv_path.exists():
        return {}
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    acc_col  = next((c for c in df.columns if "accuracy_top1" in c), None)
    loss_col = next((c for c in df.columns if "train/loss" in c), None)

    out = {"epochs_run": len(df)}
    if acc_col:
        out["best_epoch"] = int(df[acc_col].idxmax()) + 1
        out["val_acc_first"] = float(df[acc_col].iloc[0])
        out["val_acc_last"] = float(df[acc_col].iloc[-1])
        out["val_acc_best"] = float(df[acc_col].max())
    if loss_col:
        out["train_loss_first"] = float(df[loss_col].iloc[0])
        out["train_loss_last"] = float(df[loss_col].iloc[-1])
        out["loss_drop"] = out["train_loss_first"] - out["train_loss_last"]
    return out

print("=" * 74)
print("PREPARING FOREGROUND MASKS")
print("=" * 74)
MASKS = get_masks(DATA_SRC / "test", cache_path=CACHE_DIR / f"masks_fold{FOLD}.pt")

if not REF_CSV.exists():
    print("\nREFERENCE: un-finetuned partial baseline")
    ref = evaluate_ebn_model(
        model_path=str(BASE_CKPT), dataset_path=str(DATA_SRC),
        masks=MASKS, split="test", imgsz=IMGSZ, tau_percentile=TAU_PCT,
    )
    pd.DataFrame([{"config": "none", "budget": 0, "seed": -1,
                   "run_name": "partial_baseline", **ref}]).to_csv(REF_CSV, index=False)
REFERENCE = pd.read_csv(REF_CSV).iloc[0]
print(f"  reference: acc={REFERENCE['accuracy']:.4f}  QWK={REFERENCE['QWK']:.4f}  "
      f"macroF1={REFERENCE['f1_macro']:.4f}  BSR={REFERENCE['BSR']:.4f}")

if RAW_CSV.exists():
    master_df = pd.read_csv(RAW_CSV)
    records = master_df.to_dict("records")
    print(f"\nResuming: {len(records)} of "
          f"{len(CONFIGS) * len(BUDGETS) * len(SEEDS)} runs recorded")
else:
    master_df = pd.DataFrame()
    records = []


def already_done(config, budget, seed):
    if master_df.empty:
        return False
    hit = master_df[
        (master_df["config"] == config)
        & (master_df["budget"] == budget)
        & (master_df["seed"] == seed)
    ]
    return not hit.empty

total = len(CONFIGS) * len(BUDGETS) * len(SEEDS)
done = 0

for config in CONFIGS:
    for budget in BUDGETS:
        for seed in SEEDS:
            done += 1

            if already_done(config, budget, seed):
                print(f"[skip {done}/{total}] {config} | k={budget} | seed {seed}")
                continue

            print("\n" + "=" * 74)
            print(f"[{done}/{total}]  config={config}  k={budget}  seed={seed}")
            print("=" * 74)

            set_seed(seed)

            data_dir = prepare_finetune_dataset(config, budget, verbose=True)
            verify_no_leakage(data_dir)
            data_digest, n_files = digest_tree(data_dir)
            print(f"      dataset digest {data_digest} ({n_files} files)")

            run_name = f"{config}_k{budget}_seed{seed}"
            local_dir = LOCAL_RUNS / run_name
            if local_dir.exists():
                shutil.rmtree(local_dir)

            model = YOLO(str(BASE_CKPT))

            t0 = time.time()
            model.train(
                data=str(data_dir),
                project=str(LOCAL_RUNS),
                name=run_name,
                exist_ok=True,
                epochs=EPOCHS,
                patience=PATIENCE,
                imgsz=IMGSZ,
                batch=BATCH,
                workers=WORKERS,
                seed=seed,
                deterministic=True,
                freeze=FREEZE,
                optimizer=OPTIMIZER,
                lr0=LR0,
                lrf=LRF,
                cos_lr=COS_LR,
                weight_decay=WEIGHT_DECAY,
                warmup_epochs=WARMUP_EPOCHS,
                warmup_bias_lr=WARMUP_BIAS_LR,
                warmup_momentum=WARMUP_MOMENTUM,
                dropout=DROPOUT,
                erasing=0.0,
                fliplr=0.5,
                flipud=0.5,
                degrees=15.0,
                hsv_h=0.028,
                hsv_s=0.2,
                hsv_v=0.2,
            )
            train_minutes = (time.time() - t0) / 60.0

            curve = read_training_curve(local_dir)

            metrics = evaluate_ebn_model(
                model_path=str(local_dir / "weights" / "last.pt"),
                dataset_path=str(DATA_SRC), masks=MASKS, split="test",
                imgsz=IMGSZ, tau_percentile=TAU_PCT,
            )

            extra = {}
            if EVAL_BEST_TOO:
                bm = evaluate_ebn_model(
                    model_path=str(local_dir / "weights" / "best.pt"),
                    dataset_path=str(DATA_SRC), masks=MASKS, split="test",
                    imgsz=IMGSZ, tau_percentile=TAU_PCT, verbose=False,
                )
                extra = {f"bestckpt_{k}": v for k, v in bm.items()
                         if k in ("accuracy", "balanced_accuracy", "QWK",
                                  "f1_macro", "FFP", "BFP", "BSR")}

            drive_dir = RUNS_DIR / run_name
            if drive_dir.exists():
                shutil.rmtree(drive_dir)
            drive_dir.mkdir(parents=True, exist_ok=True)
            if SAVE_WEIGHTS:
                (drive_dir / "weights").mkdir(exist_ok=True)
                shutil.copy2(local_dir / "weights" / "last.pt",
                             drive_dir / "weights" / "last.pt")
            for f in ("results.csv", "args.yaml"):
                if (local_dir / f).exists():
                    shutil.copy2(local_dir / f, drive_dir / f)

            records.append({
                "config": config, "budget": budget, "c_total": budget * len(CLASSES),
                "seed": seed, "run_name": run_name,
                "base_ckpt": str(BASE_CKPT), "data_digest": data_digest,
                "optimizer": OPTIMIZER, "lr0": LR0, "freeze": FREEZE,
                "eval_ckpt": "last",
                "train_minutes": round(train_minutes, 2),
                **curve, **metrics, **extra,
            })

            master_df = pd.DataFrame(records)
            master_df.to_csv(RAW_CSV, index=False)

            if curve.get("loss_drop", 1.0) < 0.01:
                print("      [!] train loss barely moved -- if this repeats on the "
                      "next few runs, stop and raise LR0.")

            print(f"  saved -> {RAW_CSV.name}  ({len(records)}/{total} runs)  "
                  f"[{train_minutes:.1f} min, ETA "
                  f"{(total - len(records)) * train_minutes / 60:.1f} h]")

            shutil.rmtree(local_dir, ignore_errors=True)

print("\n" + "=" * 74)
print("BUILDING SUMMARY")
print("=" * 74)

master_df = pd.read_csv(RAW_CSV)
present = [c for c in METRIC_COLS if c in master_df.columns]


def describe(vals):
    n = len(vals)
    mean = float(np.mean(vals))
    sd = float(np.std(vals, ddof=1)) if n > 1 else 0.0
    sem = sd / np.sqrt(n) if n > 1 else 0.0
    ci = sem * stats.t.ppf(0.975, n - 1) if n > 1 else 0.0
    return {
        "n": n, "mean": mean, "std": sd, "sem": sem,
        "ci95_low": mean - ci, "ci95_high": mean + ci,
        "min": float(np.min(vals)), "q1": float(np.percentile(vals, 25)),
        "median": float(np.median(vals)), "q3": float(np.percentile(vals, 75)),
        "max": float(np.max(vals)), "range": float(np.max(vals) - np.min(vals)),
        "cv_pct": (sd / mean * 100) if mean else np.nan,
    }


summary_rows = []
for (config, budget), group in master_df.groupby(["config", "budget"]):
    for metric in present:
        vals = pd.to_numeric(group[metric], errors="coerce").dropna().values
        if len(vals):
            summary_rows.append({"config": config, "budget": budget,
                                 "metric": metric, **describe(vals)})
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_CSV, index=False)

byconf_rows = []
for config, group in master_df.groupby("config"):
    for metric in present:
        vals = pd.to_numeric(group[metric], errors="coerce").dropna().values
        if len(vals):
            byconf_rows.append({"config": config, "metric": metric, **describe(vals)})
pd.DataFrame(byconf_rows).to_csv(BYCONF_CSV, index=False)

delta_rows = []
ctrl = master_df[master_df["config"] == CONTROL].set_index(["budget", "seed"])

for config in CONFIGS:
    if config == CONTROL:
        continue
    arm = master_df[master_df["config"] == config].set_index(["budget", "seed"])
    for budget in BUDGETS:
        idx = [i for i in arm.index if i in ctrl.index and i[0] == budget]
        if len(idx) < 2:
            continue
        for metric in present:
            a = pd.to_numeric(arm.loc[idx, metric], errors="coerce")
            b = pd.to_numeric(ctrl.loc[idx, metric], errors="coerce")
            d = (a - b).dropna().values
            if len(d) < 2:
                continue

            t, pv = stats.ttest_1samp(d, 0.0)
            lo, hi = stats.t.interval(0.95, len(d) - 1,
                                      loc=d.mean(), scale=stats.sem(d))
            delta_rows.append({
                "config": config, "budget": budget, "metric": metric,
                "n_pairs": len(d), "mean_delta": float(d.mean()),
                "ci95_low": float(lo), "ci95_high": float(hi),
                "p_value": float(pv),
                "cohen_d": float(d.mean() / d.std(ddof=1)) if d.std(ddof=1) else np.nan,
                "significant_at_05": bool(pv < 0.05),
            })

delta_df = pd.DataFrame(delta_rows, columns=DELTA_COLS)
delta_df.to_csv(DELTA_CSV, index=False)

pivot = summary_df.copy()
pivot["value"] = pivot.apply(lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1)
pivot = pivot.pivot_table(index=["config", "budget"], columns="metric",
                          values="value", aggfunc="first")
pivot = pivot.reindex(columns=[m for m in METRIC_COLS if m in pivot.columns])
pivot.to_csv(PIVOT_CSV)

print(f"\n  raw       -> {RAW_CSV.name}")
print(f"  summary   -> {SUMMARY_CSV.name}")
print(f"  by config -> {BYCONF_CSV.name}")
print(f"  deltas    -> {DELTA_CSV.name}   (paired vs {CONTROL})")
print(f"  report    -> {PIVOT_CSV.name}")
print(f"  reference -> {REF_CSV.name}")

print("\n" + "=" * 100)
print("CONVERGENCE CHECK")
print("=" * 100)
if "best_epoch" in master_df.columns:
    print(f"  best_epoch:  median {master_df['best_epoch'].median():.0f}, "
          f"range {master_df['best_epoch'].min():.0f}-{master_df['best_epoch'].max():.0f}, "
          f"{(master_df['best_epoch'] <= 1).sum()}/{len(master_df)} stuck at 1")
if "loss_drop" in master_df.columns:
    print(f"  loss_drop:   mean {master_df['loss_drop'].mean():.4f}, "
          f"min {master_df['loss_drop'].min():.4f}")
if {"val_acc_last", "val_acc_best"}.issubset(master_df.columns):
    gap = (master_df["val_acc_best"] - master_df["val_acc_last"]).mean()
    print(f"  val gap:     best - last = {gap:.4f}  "
          f"(large positive => last.pt is overfitting; use bestckpt_* columns)")

print("\n" + "=" * 100)
print(f"FINETUNING RESULTS  (mean ± std over {len(SEEDS)} seeds, fold {FOLD}, last.pt)")
print(f"reference (no finetuning):  acc {REFERENCE['accuracy']:.4f}   "
      f"QWK {REFERENCE['QWK']:.4f}   macroF1 {REFERENCE['f1_macro']:.4f}   "
      f"BSR {REFERENCE['BSR']:.4f}")
print("=" * 100)
headline = ["accuracy", "balanced_accuracy", "QWK", "f1_macro"]
print(f"{'Config':<24}{'k':>4}" + "".join(f"{m:>18}" for m in headline))
print("-" * 100)
for config in CONFIGS:
    for budget in BUDGETS:
        row = f"{config:<24}{budget:>4}"
        for metric in headline:
            s = summary_df[(summary_df["config"] == config)
                           & (summary_df["budget"] == budget)
                           & (summary_df["metric"] == metric)]
            row += (f"{s['mean'].iat[0]:>11.4f}±{s['std'].iat[0]:.3f}"
                    if not s.empty else f"{'n/a':>18}")
        print(row)

print("\n" + "=" * 100)
print(f"PAIRED DELTAS vs {CONTROL}  (same budget and seed; QWK)")
print("=" * 100)
if delta_df.empty:
    print(f"  (none -- needs >=2 seeds per config/budget; SEEDS has {len(SEEDS)})")
else:
    print(f"{'Config':<24}{'k':>4}{'delta':>10}{'95% CI':>22}{'p':>10}{'sig':>6}")
    print("-" * 100)
    for config in CONFIGS:
        if config == CONTROL:
            continue
        for budget in BUDGETS:
            s = delta_df[(delta_df["config"] == config) & (delta_df["budget"] == budget)
                         & (delta_df["metric"] == "QWK")]
            if s.empty:
                continue
            print(f"{config:<24}{budget:>4}{s['mean_delta'].iat[0]:>+10.4f}"
                  f"{f'[{s.ci95_low.iat[0]:+.3f}, {s.ci95_high.iat[0]:+.3f}]':>22}"
                  f"{s['p_value'].iat[0]:>10.3f}"
                  f"{('yes' if s['significant_at_05'].iat[0] else 'no'):>6}")
print("=" * 100)

PREPARING FOREGROUND MASKS
  Loaded 88 cached masks from masks_fold2.pt

REFERENCE: un-finetuned partial baseline


    acc=0.5795  QWK=0.3817  macroF1=0.5014  FFP=0.4782  BFP=0.7400  BSR=0.7605
  reference: acc=0.5795  QWK=0.3817  macroF1=0.5014  BSR=0.7605

[1/90]  config=baseline_clean  k=15  seed=42
Global seed set to: 42
      train 183 ([61, 61, 61]), added [15, 15, 15] per grade
      dataset digest 0ffe746b6a5c3da7 (216 files)
Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/dataset_finetune, degrees=15.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.15, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.0, exist_ok=True, fliplr=0.5, flipu

    acc=0.6023  QWK=0.4877  macroF1=0.5125  FFP=0.7261  BFP=0.5540  BSR=0.4326


  saved -> finetune_raw.csv  (1/90 runs)  [0.9 min, ETA 1.3 h]

[2/90]  config=baseline_clean  k=15  seed=1337
Global seed set to: 1337
      train 183 ([61, 61, 61]), added [15, 15, 15] per grade
      dataset digest 0ffe746b6a5c3da7 (216 files)
Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/dataset_finetune, degrees=15.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.15, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.0, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=9, hsv_h=0.028, hsv_s=0.2, h

    acc=0.5909  QWK=0.4580  macroF1=0.5247  FFP=0.7052  BFP=0.6123  BSR=0.4629


...

    acc=0.6250  QWK=0.6224  macroF1=0.5875  FFP=0.6845  BFP=0.5881  BSR=0.5081


  saved -> finetune_raw.csv  (90/90 runs)  [1.0 min, ETA 0.0 h]

BUILDING SUMMARY


invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
Precision loss occurred in moment calculation due to catastrophic cancellation. This 


  raw       -> finetune_raw.csv
  summary   -> finetune_summary.csv
  by config -> finetune_by_config.csv
  deltas    -> finetune_deltas.csv   (paired vs baseline_clean)
  report    -> finetune_report.csv
  reference -> finetune_reference.csv

CONVERGENCE CHECK
  best_epoch:  median 4, range 1-21, 42/90 stuck at 1
  loss_drop:   mean 0.5011, min 0.4080
  val gap:     best - last = 0.1333  (large positive => last.pt is overfitting; use bestckpt_* columns)

FINETUNING RESULTS  (mean ± std over 5 seeds, fold 2, last.pt)
reference (no finetuning):  acc 0.5795   QWK 0.3817   macroF1 0.5014   BSR 0.7605
Config                     k          accuracy balanced_accuracy               QWK          f1_macro
----------------------------------------------------------------------------------------------------
baseline_clean            15     0.5932±0.010     0.5058±0.005     0.4692±0.013     0.5197±0.006
baseline_clean            30     0.6909±0.005     0.6256±0.003     0.5874±0.019     0.6188±0.00

Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
invalid value encountered in multiply
invalid value encountered in multiply
Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
invalid value encountered in multiply
invalid value encountered in multiply
Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply
invalid value encountered in multiply


# End